# HTML Filter Rules
Sample reference: `customer_dashboard.html`

---

## Phase 0 — Pre-strip (remove entire subtrees before any traversal)

Unconditionally remove these before doing anything else. They are library artifacts or rendering noise that will never be graph nodes.

| Selector | Reason |
|---|---|
| `span.select2-container` | Select2 visual shadow of a real `<select>` |
| `b[role='presentation']` | Select2 arrow icon, zero meaning |
| `div.dataTables_scrollHead` | DataTables duplicate fixed header |
| `div.dataTables_sizing` | DataTables invisible sizing ghost |
| `div.dataTables_scrollHeadInner` | Same, inner wrapper |
| `div.loading`, `div.spinner` | Loading overlays, not navigable |
| `div.daterangepicker` | Library-appended dropdown clones at body level |
| `div.calendar-tooltip` | Library tooltip container |
| `div.clear` | Bootstrap clearfix, purely layout |
| `script`, `style` | Code and styles, not UI |
| `input[type='hidden']` | Hidden state carriers, not visible or navigable |
| `[aria-hidden='true']` | Accessibility-hidden, not user-facing |

---

## Phase 1 — Element KEEP rules (whitelist)

An element is kept if it satisfies **any** of the following:

### 1. Has own direct text
The element has a non-whitespace text node as a **direct child** (not inherited from descendants).

```
<h1 class="title">Performa Kehadiran Karyawan</h1>  ✓ keep
<div class="row">...</div>                           ✗ no own text
```

### 2. Is a semantic interactive tag
Always keep regardless of attributes:
- `<button>` — interactive by nature even without onclick
- `<input>` (excluding `type="hidden"`)
- `<select>` (excluding `.select2-hidden-accessible` — those are the originals duplicated by Select2; actually **keep** the hidden-accessible ones since they hold the real `id`, **drop** the `span.select2-container` wrapper instead)
- `<textarea>`
- `<form>` — groups inputs, has action/id

### 3. Has `onclick` attribute
Any tag. `onclick` means user interaction is directly wired.

### 4. Is an `<a>` with meaningful href or onclick
- Keep if `href` is not bare `#` (e.g. real path, `javascript:void(0)`, `javascript:;`)
- Keep if `href="#"` but has `data-page` or `onclick`
- Drop `<a href="#">Dummy</a>` breadcrumb roots that are purely structural with no action

### 5. Has an `id` attribute
JS (and your highlighter) can target it by id. Also, elements with ids are often toggle targets (e.g., `id="div_emp_list"` gets `style="display:none"` toggled).

### 6. Has a meaningful `data-*` attribute
Keep if has any of: `data-toggle`, `data-target`, `data-page`, `data-index`, `data-original-title`, `data-dismiss`, `data-href`.

### 7. Is a heading tag
`<h1>` through `<h6>` — they define the semantic section title.

### 8. Is an icon element with a semantic class
- `<i class="bi bi-*">` or `<i class="fa fa-*">` — keep tag + class, collapse children
- `<svg class="bi bi-*">` — keep as `<svg class="..."/>`, strip all `<path>` children entirely

### 9. Is table structure
Keep: `<table>`, `<thead>`, `<tbody>`, `<tr>`, `<th>`, `<td>`.  
Table columns and rows describe what data a page manages — critical for navigation context.

### 10. Is pagination or list structure
Keep `<ul>`, `<li>` inside `.pagination`, `.breadcrumb` — these are functionally meaningful, not just lists.

### 11. Has a non-layout semantic class (for divs/spans)
A `div` or `span` is kept if it has at least one class that is **not** in the layout class blocklist below.  
This covers cases like `div.dashboard-card`, `div.quota-card`, `div.btn-show-container`, `div.filter-date` which JS or the agent may reference by class.

**Layout class blocklist** (drop-only divs if ALL classes are in this list):
```
row, col, col-md-*, col-lg-*, col-sm-*, col-xs-*, col-xl-*,
d-flex, align-items-*, justify-content-*, flex-wrap,
content-w, layout-w, all-wrapper, content-box, element-wrapper,
element-box-tp, right-wrapper, left-wrapper,
top, bottom, clear, selection, dataTables_scroll, dataTables_scrollBody
```

---

## Phase 2 — DROP rules (applied after whitelist)

Even if an element passed the whitelist, drop it if:

1. **Empty with no id and no data-* and not interactive** — e.g. `<div id="notif_duedate_mobile"></div>` has an id so keep it; but `<p class="subtitle"></p>` with no content → drop.

2. **`<canvas>`** — chart rendering surface, not interactable for navigation. Drop entirely.

3. **`<img>`** — display only. Drop unless it has `onclick`.

4. **`<option>` elements** — keep only the parent `<select>`, not individual options. Options clutter the graph without adding navigational value (the agent doesn't need to know the option values to navigate).  
   **Exception**: keep `<option>` only if you need them for the knowledge graph embedding context (debatable — decide at embedding stage).

5. **Wrapper div that is a solo-child wrapper of a single interactive element AND has no id AND has no semantic class** — e.g. `<div class="form-calendar"><input .../></div>`. The `form-calendar` class is semantic so it stays; a `<div class="col-lg-2"><select .../></div>` dies.

6. **`<span>` used purely as a Select2 rendered text display** — e.g. `<span class="select2-selection__rendered">`. These are pre-stripped in Phase 0 anyway.

---

## Phase 3 — Attribute cleaning

After deciding to keep an element, clean its attributes:

1. **Strip IP from all URLs**: remove `http://107.155.65.156:81` in `href`, `src`, `action`, and inside `onclick` string content.  
   e.g. `onclick="loadPage('http://107.155.65.156:81/customer/setting/billing_info', true)"` → `onclick="loadPage('/customer/setting/billing_info', true)"`

2. **Strip purely visual inline styles**: remove `style` attribute entirely.  
   Exception: keep `style="display:none"` — it signals a hidden/toggled region.

3. **Strip select2/datatable internal attributes**: `data-select2-id`, `aria-controls`, `aria-labelledby`, `aria-expanded`, `aria-haspopup`, `aria-disabled`, `aria-readonly`, `tabindex`, `dir`, `role` (except `role="grid"` on tables).

4. **Strip `selected=""` from `<option>`** — dynamic state, not structural.

5. **SVG collapse**: replace full `<svg class="...">...</svg>` with `<svg class="..."/>`.

---

## Summary decision table

| Element | Decision | Reason |
|---|---|---|
| `div.row`, `div.col-*` | DROP if no semantic class | Pure layout |
| `div.btn-show-container` | KEEP | Semantic class, JS-targetable |
| `div.dashboard-card` | KEEP | Semantic class, meaningful section |
| `div.filter-date#filter_date` | KEEP | Has id |
| `select.select2-hidden-accessible` | KEEP | Real element, has id |
| `span.select2-container` | DROP (Phase 0) | Visual shadow |
| `button` (no onclick) | KEEP | Interactive by tag |
| `<a href="#">Dummy</a>` | DROP | Bare # with no action |
| `<a href="#" data-page="-1">` | KEEP | Has data-page |
| `<canvas id="attendanceChart">` | DROP | Chart surface |
| `<img>` | DROP | Display only |
| `<i class="bi bi-star">` | KEEP | Semantic icon |
| `<svg class="bi bi-arrow-repeat">` | KEEP, collapse paths | Semantic icon |
| `<script>` | DROP (Phase 0) | Code |
| `div.daterangepicker` at body | DROP (Phase 0) | Library artifact |
| `input[type='hidden']` | DROP (Phase 0) | Not visible/navigable |
| `<p class="subtitle"></p>` (empty) | DROP | No content, no id |
| `<div id="notif_duedate_mobile">` (empty) | KEEP | Has id — JS target |


In [3]:
from bs4 import BeautifulSoup, NavigableString, Tag
import re, copy

# ── Phase 0: Pre-strip selectors ──────────────────────────────────────────────
PRESTRIP_SELECTORS = [
    "span.select2-container",
    "b[role='presentation']",
    "div.dataTables_scrollHead",
    "div.dataTables_scrollHeadInner",
    "div.dataTables_sizing",
    "div.loading",
    "div.spinner",
    "div.daterangepicker",
    "div.calendar-tooltip",
    "div.clear",
    "script",
    "style",
    "input[type='hidden']",
    # NOTE: NOT [aria-hidden='true'] globally — Select2 sets aria-hidden on the real <select>
    # to suppress screen reader duplication. We want that select. Instead we only strip
    # the span.dropdown-wrapper that Select2 appends (already covered by span.select2-container).
]

# ── Layout class blocklist (div/span kept only if it has ≥1 class NOT in here) ─
LAYOUT_CLASSES = {
    "row", "col", "top", "bottom", "selection",
    "d-flex", "flex-wrap", "float-left", "float-right",
    "content-w", "content-i", "content-box", "layout-w",
    "all-wrapper", "element-wrapper", "element-box-tp",
    "right-wrapper", "left-wrapper",
    "dataTables_scroll", "dataTables_scrollBody",
    "dataTables_wrapper",
}
LAYOUT_CLASS_PREFIXES = (
    "col-", "align-items-", "justify-content-", "p-", "m-",
    "pb-", "pt-", "mb-", "mt-", "px-", "py-", "mx-", "my-",
)

# ── Meaningful data-* attributes ──────────────────────────────────────────────
MEANINGFUL_DATA_ATTRS = {
    "data-toggle", "data-target", "data-page", "data-index",
    "data-original-title", "data-dismiss", "data-href",
}

# ── Attributes to strip from kept elements ────────────────────────────────────
STRIP_ATTRS = {
    "data-select2-id", "aria-controls", "aria-labelledby",
    "aria-expanded", "aria-haspopup", "aria-disabled", "aria-readonly",
    "aria-describedby", "aria-label", "aria-live", "aria-hidden",
    "tabindex", "dir", "colspan", "rowspan",
}
STRIP_ROLE_VALUES = {"combobox", "textbox", "presentation"}  # keep "grid"

IP_PATTERN = re.compile(r'https?://[\d\.]+(?::\d+)?')


def is_layout_only(tag: Tag) -> bool:
    """True if a div/span has ONLY layout/grid classes."""
    classes = set(tag.get("class", []))
    if not classes:
        return True  # classless div with no id → layout
    for cls in classes:
        is_layout = cls in LAYOUT_CLASSES or any(cls.startswith(p) for p in LAYOUT_CLASS_PREFIXES)
        if not is_layout:
            return False  # found at least one semantic class
    return True


def has_own_text(tag: Tag) -> bool:
    """True if the tag has a non-whitespace direct text child."""
    for child in tag.children:
        if isinstance(child, NavigableString) and child.strip():
            return True
    return False


def has_meaningful_data(tag: Tag) -> bool:
    attrs = set(tag.attrs.keys())
    return bool(attrs & MEANINGFUL_DATA_ATTRS)


def should_keep(tag: Tag) -> bool:
    name = tag.name

    # Always drop canvas, img (unless onclick), option
    if name == "canvas":
        return False
    if name == "img":
        return bool(tag.get("onclick"))
    if name == "option":
        return False

    # Semantic interactive tags
    if name in ("button", "textarea", "form"):
        return True
    if name == "input":
        return tag.get("type", "text").lower() != "hidden"
    if name == "select":
        return True  # Keep the real select (select2-container span already stripped in Phase 0)

    # Anchors
    if name == "a":
        href = tag.get("href", "")
        onclick = tag.get("onclick", "")
        has_data = has_meaningful_data(tag)
        if onclick or has_data:
            return True
        if href and href != "#":
            return True
        return False

    # Headings
    if name in ("h1", "h2", "h3", "h4", "h5", "h6"):
        return True

    # Table structure
    if name in ("table", "thead", "tbody", "tr", "th", "td"):
        return True

    # Pagination / breadcrumb lists
    if name in ("ul", "li"):
        own_classes = set(tag.get("class", []))
        parent = tag.parent
        parent_classes = set(parent.get("class", [])) if isinstance(parent, Tag) else set()
        if "pagination" in own_classes or "breadcrumb" in own_classes:
            return True
        if "pagination" in parent_classes or "breadcrumb" in parent_classes:
            return True

    # Icons
    if name == "i":
        classes = tag.get("class", [])
        if any(c.startswith("bi") or c.startswith("fa") or c.startswith("os-icon") or c.startswith("icon-") for c in classes):
            return True

    # SVG with semantic class
    if name == "svg":
        return bool(tag.get("class"))

    # Has id → keep
    if tag.get("id"):
        return True

    # Has meaningful data-* → keep
    if has_meaningful_data(tag):
        return True

    # Has onclick
    if tag.get("onclick"):
        return True

    # Has own direct text
    if has_own_text(tag):
        return True

    # Div/span with semantic class
    if name in ("div", "span", "section", "article", "aside", "nav"):
        if not is_layout_only(tag):
            return True

    return False


def clean_attrs(tag: Tag):
    """Strip noisy attributes in-place, clean URLs."""
    for attr in list(tag.attrs.keys()):
        if attr in STRIP_ATTRS:
            del tag.attrs[attr]
        elif attr == "role" and tag.attrs.get("role") in STRIP_ROLE_VALUES:
            del tag.attrs["role"]
        elif attr == "style":
            style_val = tag.attrs[attr]
            if "display:none" in style_val.replace(" ", "") or "display: none" in style_val:
                tag.attrs[attr] = "display: none"
            else:
                del tag.attrs[attr]

    # Strip IP from url-bearing attributes and onclick strings
    for attr in ("href", "src", "action", "onclick", "menu_url"):
        if attr in tag.attrs:
            tag.attrs[attr] = IP_PATTERN.sub("", tag.attrs[attr])

    # Collapse SVG: remove all children (path elements)
    if tag.name == "svg":
        tag.clear()


def prestrip(soup: BeautifulSoup):
    """Phase 0: remove entire subtrees unconditionally."""
    for selector in PRESTRIP_SELECTORS:
        for el in soup.select(selector):
            el.decompose()


def filter_tree(tag: Tag):
    """
    Recursively filter a tag tree.
    Unwraps tags that should not be kept but have useful children.
    Removes tags that should not be kept and have no useful children.
    """
    if not isinstance(tag, Tag):
        return

    # Recurse into children first (bottom-up)
    for child in list(tag.children):
        if isinstance(child, Tag):
            filter_tree(child)

    keep = should_keep(tag)
    if not keep:
        # Check if there's anything worth preserving inside
        remaining = [c for c in tag.children
                     if isinstance(c, Tag) or (isinstance(c, NavigableString) and c.strip())]
        if remaining:
            tag.unwrap()  # promote children to parent
        else:
            tag.decompose()  # nothing inside, remove entirely
        return

    clean_attrs(tag)


def process_html(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")
    prestrip(soup)

    body = soup.find("body") or soup
    filter_tree(body)

    return soup.prettify()


# ── Quick test on dashboard ────────────────────────────────────────────────────
INPUT_PATH = "../data/cleaned_html/customer_dashboard.html"
OUTPUT_PATH = "../data/filtered_html/customer_dashboard.html"

import os
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

with open(INPUT_PATH, encoding="utf-8") as f:
    raw = f.read()

result = process_html(raw)

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    f.write(result)

print(f"Done. Lines: {len(raw.splitlines())} → {len(result.splitlines())}")


Done. Lines: 1108 → 782


In [4]:
# Preview first 120 lines of filtered output
with open(OUTPUT_PATH, encoding="utf-8") as f:
    lines = f.readlines()

print("".join(lines[:120]))


<html>
 <div class="all-wrapper with-side-panel solid-bg-all">
  <div class="content-w new pb-4">
   <div id="notif_duedate_mobile">
   </div>
   <div id="content">
    <div class="content-w custom-container">
     <div class="row custom-row">
      <ul class="breadcrumb custom-breadcrumb">
       <li class="breadcrumb-item">
        Dummy
       </li>
       <li class="breadcrumb-item">
        <span>
         Dashboard
        </span>
       </li>
      </ul>
      <i class="bi bi-star" id="btn_favorite" menu_id="0.1" menu_name="Dashboard" menu_url="/customer/dashboard" onclick="favorite(this)">
      </i>
     </div>
    </div>
    <div class="dashboard-content">
     <div class="dashboard-card attendance-report">
      <div class="card-header">
       <h1 class="title">
        Performa Kehadiran Karyawan
       </h1>
       <select class="select select2-hidden-accessible" id="month-select">
        Mei 2025
        Juni 2025
        Juli 2025
        Agustus 2025
        September

notes on the previous rule:

1. i dont think that removing the style (the color specifically) is a good idea. this will be useful for the navigation. what if user asks "what is this red button im looking at?". the style (the color) will be placed in the element description later, that is created by LLM. 